# Homework 02: SQL Mini-Project with Sakila
## Joins, Aggregation, CTEs, Window Functions, and Performance

**Due date:** _April 3, 2026 by 11:59 PM_

**Points:** 100 total

> **Name (FIRST and LAST)** — Justin Wijaya

- - -

## Overview and Instructions

**Context:** This homework builds directly on our relational database + SQL module (Days 11–19). You will use a Dockerized MySQL database (the Sakila sample database you set up previously) and a Jupyter notebook to carry out a small, portfolio-ready SQL analysis.

You will:
- Connect to a Dockerized MySQL database from this notebook.
- Design and implement several non-trivial SQL queries using Sakila.
- Use joins, grouped aggregation with `HAVING`, CTEs, window functions, and a brief performance check.
- Practice explicit validation habits (row counts, sanity checks, cross-checks).
- Produce at least one manager-style report table that could stand alone in a portfolio.
- Reflect on your SQL reasoning and debugging process.
- Log any use of AI tools in an **AI Audit Log** section.
- Create a GitHub repository to share your work.

**If you get stuck, have questions, or run into any issues as you work through this assignment**, remember to consult the PCAs, ICAs, slides from class for the **Day 11-19 class periods** -- the slides in particular often have worked examples of queries that are similar to what you'll need to write for this assignment. You are also encouraged to **come to office hours** to get direct support from your instructors! If you can't find out office hours information or need to schedule by appoint, **send us an email or catch one of us at the end of class**.

**Submission:**
- Push this completed notebook (and any small helper files) to a new private GitHub repo called `cmse492-hw02-yourMSUNetID` (make sure you use your actual MSU NetID!). **Details for this process are include in Part 10 below.**
- Submit the GitHub URL via the provided Microsoft form.
- Upload the executed notebook (with outputs) to D2L as a backup.

This notebook is designed to be a **self-contained artifact**: someone with access to your Docker image and credentials should be able to rerun your analysis and understand what you did, why you did it, and how you validated it.

**Grading breakdown:** See section headings for point values.

<div class="alert alert-attention">

**Note about AI tools:** As discussed at the beginning of the course, you are allowed to use AI tools (e.g., ChatGPT, Copilot) in responsible, transparent, and ethical ways. For this particular assignment, if you end up using AI tools for assignment, but you will be expected to document your usage in an "AI Audit Log" section near the end of this notebook. See the details in the "AI Audit Log" section below.

</div>


- - -

## 0. Setup and Database Connection

In this section you will:
- Confirm your Dockerized MySQL + Sakila setup is working.
- Establish a connection from Python to the database using `sqlalchemy`.
- Define a small helper function to run SQL and get results as a `pandas` DataFrame (make sure you look at this function so that you understand how it works and can use it in the assignment!)

**Even though you are being provided with the code cell necessary to connect to the database, you should make sure you carefully review the code and understand how it is working.** If you end up needing to connect to a SQL database on your own in the future, you should know how to do so.

**Remember**: Before you try to connect to the database, make sure your Docker container is running and that the Sakila database is properly set up inside it. If you don't recall how to do this, review the instructions from the Day 17 content and get that set up first.

If you run into an issue with this part of the assignment, contact your instructors _as soon as possible_ so we can help you get it resolved. You will not be able to complete the rest of the assignment without a working database connection, so this is a critical first step.


In [1]:
import os
import pandas as pd
from sqlalchemy import create_engine, text

# Credentials for the local Docker MySQL container (must match docker-compose.yml)
uname = 'sakila'
pwd = 'p_ssW0rd'
hname = 'localhost'
dbname = 'sakila'

engine = create_engine(f'mysql+pymysql://{uname}:{pwd}@{hname}/{dbname}')
connection = engine.connect()

def run_sql(query, params=None):
    """Run a SQL query and return a pandas DataFrame.
    `query` can be a string; `params` is an optional dict for parameterized queries.
    """
    with engine.connect() as conn:
        result = conn.execute(text(query), params or {})
        df = pd.DataFrame(result.fetchall(), columns=result.keys())
    return df

# Quick test to confirm that you can connect and query the database
test_df = run_sql("SELECT * FROM film LIMIT 5;")
test_df.head()

,film_id,title,description,release_year,language_id,original_language_id,rental_duration,rental_rate,length,replacement_cost,rating,special_features,last_update
0,1,ACADEMY DINOSAUR,A Epic Drama of a Feminist And a Mad Scientist...,2006,1,None,6,0.99,86,20.99,PG,"Deleted Scenes,Behind the Scenes",2006-02-15 05:03:42
1,2,ACE GOLDFINGER,A Astounding Epistle of a Database Administrat...,2006,1,None,3,4.99,48,12.99,G,"Trailers,Deleted Scenes",2006-02-15 05:03:42
2,3,ADAPTATION HOLES,A Astounding Reflection of a Lumberjack And a ...,2006,1,None,7,2.99,50,18.99,NC-17,"Trailers,Deleted Scenes",2006-02-15 05:03:42
3,4,AFFAIR PREJUDICE,A Fanciful Documentary of a Frisbee And a Lumb...,2006,1,None,5,2.99,117,26.99,G,"Commentaries,Behind the Scenes",2006-02-15 05:03:42
4,5,AFRICAN EGG,A Fast-Paced Documentary of a Pastry Chef And ...,2006,1,None,6,2.99,130,22.99,G,Deleted Scenes,2006-02-15 05:03:42


- - -

## 1. Framing Your Mini-Project (5 points)

Your goal is to design a **small analytic story** using the Sakila database. Think of a manager at a video rental company who wants to understand something about customers, films, inventory, stores, or revenue.

In this section, you will define:
- A brief business/analytic question or set of questions you want to answer using SQL and the audience you are imagining.
- The main tables you expect to use.
- The kinds of outputs you want (e.g., a ranked table, grouped summary, percent-of-total view).

**Note**: you might find that your question and plans evolve as you start writing SQL and seeing results. This is totally normal! Just make sure to document your initial thinking here, and then as you evolve your question and plans, make sure to update this section to reflect that evolution. Make sure to keep your initial question and plans in there as well, so that we can see how your thinking evolved.

**Another note**: You might need to spend some time exploring the Sakila schema and data to come up with a question that is interesting and feasible. You can access the DB using Adminer (via `http://localhost:8080`) to explore the schema and/or use SQL queries to poke at the data as part of your process of coming up with a good question.

If you're really struggling to come up with your "story", you can start working through the SQL exercises in the sections that follow, which require you to come up with individual questions to answer with SQL. **You might find that as you do those exercises, you see a collection of insights that your can develop into your project story.** "Reverse-engineering" your project story from the SQL exercises is a totally valid way to come up with a project question and plan!

### 1.1 Project question and plan

**Prompt:** In 1–2 paragraphs, describe your mini-project:
- What is the main question or set of questions you want to answer using SQL?
- Who is the imagined audience (e.g., store manager, marketing analyst, operations lead)?
- Which core tables do you expect to rely on (list at least 3 Sakila tables and why)?
- What kind of final table(s) or figure(s) do you expect to produce (e.g., top categories by revenue, customer segments by activity)?


> The main question to answer using SQL is what components of a film bring in the most sales in May? The imagined audience are store managers. Core tables are sales_by_film_category as it contains categories and their respective sales, film_category as it contains which films are a part of which category, and film as it contains each information on each film. I expect to produce a table that lists combinations of different aspects of the films and ranks them based on sales.
>

---

## 2. Joins and Multi-Table Queries (15 points)

In this section, you will write **at least two non-trivial join queries** that connect 2–3 tables with meaningful conditions (not just a single key lookup). For each query:
- Clearly state the question it answers.
- Sketch what a single row in the result represents.
- Write and run the SQL.
- Perform at least one validation check (row counts, spot-check rows against base tables, etc.).


### 2.1 Join Query 1

**Example idea (you may adapt this or pick a different one, do not use exactly this example):**
- "For each rental in a specified 60-day period, show customer name, film title, rental date, and store city."

Based on your question and plans from the previous section, write your own question and design for this first join query. You can use the example above as a template or inspiration, but make sure to write your own unique question and design that fits with your overall mini-project. For this first join query, you're expected to clearly define the following:

1. **Question in words**: Describe what you are trying to learn in one or two sentences.
2. **Row meaning sketch**: What does each row represent? Which columns should appear?
3. **Tables and keys**: Which tables are you joining (e.g., `rental`, `customer`, `inventory`, `film`, `store`, `address`, `city`)? On which key columns?
4. **SQL query**: Write the join with clear aliases.
5. **Validation**: Perform at least one validation check, such as:
   - Compare a row count to a simpler query.
   - Spot-check an individual record across base tables.


> **2.1.a Question and design**
> 
> 1. For each film category, show total number of rentals in May.
> 2. Each row represents a film category. The columns to appear are the film categories name and count of rentals of films under that category.
> 3. category.category_id = film_category.category_id, film_category.film_id = inventory.film_id, inventory.inventory_id = rental.inventory_id.
>

In [2]:
# 2.1.b Join Query 1 - SQL

# Now that you've defined your question and design for the first join query,
# write the SQL to implement it. Make sure to use clear aliases for tables and
# columns, and to include any necessary WHERE conditions to filter the data as
# needed for your question.

join_query_1 = """
-- TO DO: Replace this with your actual query.
-- Example(do not submit unchanged):
-- SELECT c.customer_id, c.first_name, c.last_name,
--        f.title AS film_title,
--        r.rental_date,
--        ci.city AS store_city
-- FROM rental AS r
-- JOIN inventory AS i ON r.inventory_id = i.inventory_id
-- JOIN film AS f ON i.film_id = f.film_id
-- JOIN customer AS c ON r.customer_id = c.customer_id
-- JOIN store AS s ON i.store_id = s.store_id
-- JOIN address AS a ON s.address_id = a.address_id
-- JOIN city AS ci ON a.city_id = ci.city_id
-- WHERE r.rental_date BETWEEN '2005-05-01' AND '2005-06-30';
SELECT
    c.name,
    COUNT(r.inventory_id) AS n_rentals
FROM category AS c
JOIN film_category AS fc ON c.category_id = fc.category_id
JOIN inventory AS i ON fc.film_id = i.film_id
JOIN rental AS r ON i.inventory_id = r.inventory_id
WHERE r.rental_date like "%-05-%"
GROUP BY c.name
ORDER BY n_rentals DESC;
"""

# Once you've defined your query, you can uncomment the lines below to run it and see the results.
df_join_1 = run_sql(join_query_1)
df_join_1.head()

,name,n_rentals
0,Action,87
1,Documentary,86
2,Family,85
3,Drama,85
4,Sci-Fi,84


**2.1.c Validation for Join Query 1**

Describe at least one validation check you ran and what you concluded based on it.

You might:
- Show a supporting SQL snippet in a small code cell (e.g., `COUNT(*)` comparison).
- Spot-check one ID across multiple tables and describe what you saw.


In [3]:
# Example based on the sample query above:
# check total rentals in the same date range without joins

# validation_query_1 = """
# SELECT COUNT(*) AS n_rentals_last_60_days
# FROM rental
# WHERE rental_date BETWEEN '2005-05-01' AND '2005-06-30';
# """
# run_sql(validation_query_1)

In [4]:
# Put your own validation query(ies) here and explanation in the next section.
validation_query_1 = """
SELECT COUNT(*) AS total_n_rentals
FROM rental
WHERE rental_date like "%-05-%";
"""
print("Sum of number of rentals per category:", df_join_1["n_rentals"].sum())
run_sql(validation_query_1)

Sum of number of rentals per category: 1156


,total_n_rentals
0,1156


> **_Write your validation explanation here._**
> Check if the sum of the number of rentals from the query add up to the total number of rentals in May. They do match, so number of rentals is accurate (nothing is duplicated or skipped).

### 2.2 Join Query 2

Create a **different** join that still aligns with your project theme.

**Example ideas (again, you should adapt these to your project or create your own):**
- Customer-level summary: each row is one customer with their total rentals, number of distinct categories they rent, and home city.
- Inventory-level view: each row is one film copy with film title, store, and how many times it has been rented.

Repeat the pattern as before, clearly defining:
1. Question in words.
2. Row meaning sketch.
3. Tables and keys.
4. SQL query.
5. Validation check(s).

> **2.2.a Question and design**
> 
> 1. What's the most rented film in May?
> 2. Each row is a film's title, category, length, rating, and number of rentals.
> 3. film.film_id = film_category.film_id, film_category.category_id = category.category_id, film.film_id = inventory.film_id, inventory.inventory_id = rental.inventory_id
>

In [5]:
# 2.2.b Join Query 2 - SQL
join_query_2 = """
SELECT
    f.title,
    c.name AS category,
    f.length,
    f.rating,
    COUNT(r.inventory_id) AS n_rentals
FROM film AS f
JOIN film_category AS fc ON f.film_id = fc.film_id
JOIN category AS c ON fc.category_id = c.category_id
JOIN inventory AS i ON f.film_id = i.film_id
JOIN rental AS r ON i.inventory_id = r.inventory_id
WHERE r.rental_date like "%-05-%"
GROUP BY
    f.title,
    c.name,
    f.length,
    f.rating
ORDER BY n_rentals DESC;
"""

# Once you've defined your query, you can uncomment the lines below to run it and see the results.
df_join_2 = run_sql(join_query_2)
df_join_2.head()

,title,category,length,rating,n_rentals
0,IDOLS SNATCHERS,Children,84,NC-17,5
1,BUCKET BROTHERHOOD,Travel,133,PG,5
2,ROBBERS JOON,Children,102,PG-13,5
3,LOVE SUICIDES,Horror,181,R,4
4,BOOGIE AMELIE,Music,121,R,4


**2.2.c Validation for Join Query 2**

As before, describe your validation steps and conclusions for this second join query. You can use similar validation techniques as before or come up with new ones that fit the specific question and data you are working with.

In [6]:
# As appropriate, define helper validation query(ies) for Join Query 2
validation_query_2 = """
SELECT title, length, rating FROM film WHERE title = 'BUCKET BROTHERHOOD';
"""
run_sql(validation_query_2)

,title,length,rating
0,BUCKET BROTHERHOOD,133,PG


In [7]:
validation_query_2_1 = """
SELECT
    f.title,
    COUNT(*) AS n_rentals_check
FROM film f
JOIN inventory i ON f.film_id = i.film_id
JOIN rental r ON i.inventory_id = r.inventory_id
WHERE r.rental_date like '%-05-%'
GROUP BY f.film_id, f.title
ORDER BY n_rentals_check DESC;
"""
run_sql(validation_query_2_1)

,title,n_rentals_check
0,IDOLS SNATCHERS,5
1,BUCKET BROTHERHOOD,5
2,ROBBERS JOON,5
3,LOVE SUICIDES,4
4,BOOGIE AMELIE,4
...,...,...
681,SUNDANCE INVASION,1
682,SUGAR WONKA,1
683,POLLOCK DELIVERANCE,1
684,BABY HALL,1


> **_Describe your validation steps and conclusions here._**
> Check if film details are correct and if number of rentals was duplicated or not (possibly due to extra joins). Details line up (no attributing of fields to wrong record) and number of rentals is the same (no duplicates or skips), so result is accurate.

---

## 3. Grouped Aggregation with HAVING (15 points)

In this section you will:
- Write at least one grouped aggregation that uses `GROUP BY` and `HAVING`.
- Apply sanity checks (e.g., sum of group counts vs. raw counts, spot-check of a group).
- Tie the result to a question your audience might actually care about.


### 3.1 Aggregation Query with HAVING (required)

**Example ideas (adapt or design your own to fit your audience and align with your mini-project):**
- "Find film categories with at least 1000 total rentals, ordered from most to least rented."
- "Find customers who have spent at least \$X in total payments."

For your query, **document**:
1. Business question in words (1–2 sentences).
2. Row meaning and expected columns in the grouped result.
3. SQL query with `GROUP BY` and `HAVING`.
4. At least two sanity checks (e.g., compare grouped count sums to raw counts, spot-check a single group).


> **3.1.a Question and design**
> 
> 1. Find categories with an average length of at least 120 minutes.
> 2. Each row is a category and its average length.
>   

In [8]:
# 3.1.b Aggregation with GROUP BY + HAVING

# Now that you've defined your question and design for the aggregation query,
# write the SQL to implement it. Make sure to use clear aliases for tables and
# columns, to include the appropriate GROUP BY and HAVING clauses, and to filter
# the data as needed for your question.

agg_query = """
-- TODO: Replace with your actual aggregation query.
-- Example (do not submit unchanged):
-- SELECT c.name AS category_name,
--        COUNT(*) AS n_rentals
-- FROM rental AS r
-- JOIN inventory AS i ON r.inventory_id = i.inventory_id
-- JOIN film AS f ON i.film_id = f.film_id
-- JOIN film_category AS fc ON f.film_id = fc.film_id
-- JOIN category AS c ON fc.category_id = c.category_id
-- GROUP BY c.name
-- HAVING COUNT(*) >= 1000
-- ORDER BY n_rentals DESC;
SELECT
    c.name,
    AVG(f.length) AS average_length
FROM category AS c
JOIN film_category AS fc ON c.category_id = fc.category_id
JOIN film AS f ON fc.film_id = f.film_id
GROUP BY c.name
HAVING average_length >= 120
ORDER BY average_length DESC;
"""

# Once you've defined your query, you can uncomment/edit the lines below to run it and see the results.
df_agg = run_sql(agg_query)
display(df_agg) or df_agg.head()

,name,average_length
0,Sports,128.2027
1,Games,127.8361
2,Foreign,121.6986
3,Drama,120.8387


,name,average_length
0,Sports,128.2027
1,Games,127.8361
2,Foreign,121.6986
3,Drama,120.8387


**3.1.c Sanity checks for aggregation**

Describe at least **two** checks you performed. Ideas:
- Compare the sum of group counts to a raw `COUNT(*)` over a comparable subset.
- Focus on one group (e.g., one category or one customer), query its underlying rows, and verify the count.
- If applicable, use a `LEFT JOIN` to see categories with zero values and explain what you observe.


In [9]:
# As appropriate, write helper sanity-check queries for aggregation here and
# provide explanation in the next section.

check_query_1 = """
-- TODO: Replace with your actual validation query for the aggregation.
SELECT
    AVG(f.length) AS average_length
FROM film AS f;
"""
run_sql(check_query_1)

check_query_2 = """
-- TODO: Replace with your actual validation query for the aggregation.
SELECT
    c.name,
    AVG(f.length) AS average_length
FROM category c
JOIN film_category fc ON c.category_id = fc.category_id
JOIN film f ON fc.film_id = f.film_id
GROUP BY c.name
ORDER BY average_length DESC;    
"""
run_sql(check_query_2)

,name,average_length
0,Sports,128.2027
1,Games,127.8361
2,Foreign,121.6986
3,Drama,120.8387
4,Comedy,115.8276
5,Family,114.7826
6,Music,113.6471
7,Travel,113.3158
8,Horror,112.4821
9,Classics,111.6667


> **_Write your sanity-check explanation here._**
> Less than half of all categories are over 120 minutes in length, so the average length of all films should be less than 120 minutes. And it is true, so the number of categories with over 120 minutes in length is accurate.
> Computed without the having statement to see if there were other categories that weren't captured with the having statement that also have over 120 minutes average in length. There are no other categories with over 120 average minutes, so the original query is accurate. 

- - - 

## 4. Multi-Step Query Using a CTE (15 points)

Now you will use a **CTE (`WITH` clause)** to break a complex analysis into readable steps. You may reuse logic from earlier sections, but the CTE should add structure (e.g., pre-filtering, intermediate grouping) rather than just rename an existing query.

**Example patterns:**
- First CTE: compute customer-level rental counts and total payments.
- Second step: filter to active customers, compute ranks or thresholds on top.

Your CTE query must:
- Use at least one `WITH` clause.
- Do something non-trivial in the CTE (filtering, joining, grouping, etc.).
- Be accompanied by validation (e.g., run parts of the CTE separately, check a subset).

### 4.1 CTE design and explanation

Describe your CTE in words:
- What does the CTE compute?
- What does the outer query do on top of it?
- How does this decomposition make the logic clearer than a single big query would be?


> **_Write your CTE design explanation here._**
> CTE computes number of rentals in May and the total number of rentals. The outer query combines the CTEs into one table for easy comparison. This decomposition makes the actual query easier to read and understand by breaking the problem down into logical steps.

### 4.2 Writing the CTE query

Now that you have a plan for your CTE, write the SQL query that implements it.

In [10]:
# 4.2 CTE-based query

# Now that you've designed your CTE, write the SQL query that implements it.
# Make sure to use clear aliases for tables and columns, and to structure the
# CTE in a way that breaks down the logic into understandable steps.

cte_query = """
-- TODO: Replace with your CTE-based query.
WITH rentals_in_may AS (
    SELECT
        f.title,
        COUNT(r.inventory_id) AS n_rentals
    FROM film AS f
    JOIN film_category AS fc ON f.film_id = fc.film_id
    JOIN category AS c ON fc.category_id = c.category_id
    JOIN inventory AS i ON f.film_id = i.film_id
    JOIN rental AS r ON i.inventory_id = r.inventory_id
    WHERE r.rental_date LIKE '%-05-%'
    GROUP BY f.title
),
rentals_overall AS (
    SELECT
        f.title,
        COUNT(r.inventory_id) AS n_rentals
    FROM film AS f
    JOIN film_category AS fc ON f.film_id = fc.film_id
    JOIN category AS c ON fc.category_id = c.category_id
    JOIN inventory AS i ON f.film_id = i.film_id
    JOIN rental AS r ON i.inventory_id = r.inventory_id
    GROUP BY f.title
)
SELECT
    rentals_in_may.title,
    rentals_in_may.n_rentals,
    rentals_overall.n_rentals
FROM rentals_in_may
    CROSS JOIN rentals_overall
ORDER BY rentals_in_may.n_rentals DESC;
"""

# Once you've defined your query, you can uncomment the lines below to run it and see the results.
df_cte = run_sql(cte_query)
display(df_cte) or df_cte.head()

,title,n_rentals,n_rentals
0,ROBBERS JOON,5,23
1,BUCKET BROTHERHOOD,5,23
2,IDOLS SNATCHERS,5,23
3,ROBBERS JOON,5,7
4,BUCKET BROTHERHOOD,5,7
...,...,...,...
657183,DISCIPLE MOTHER,1,8
657184,REMEMBER DIARY,1,8
657185,CONFESSIONS MAGUIRE,1,8
657186,CHAMPION FLATLINERS,1,8


,title,n_rentals,n_rentals
0,ROBBERS JOON,5,23
1,BUCKET BROTHERHOOD,5,23
2,IDOLS SNATCHERS,5,23
3,ROBBERS JOON,5,7
4,BUCKET BROTHERHOOD,5,7


### 4.3 CTE validation

Run at least one validation step focused on the **intermediate CTE logic**, such as:
- Select a subset of the CTE output (e.g., `LIMIT 10`) and manually compare to base tables.
- Recompute a simpler version of a metric (e.g., total payments) for one customer and confirm it matches the CTE result.

In [11]:
# As appropriate, write helper queries to inspect the CTE result or base tables
check_cte_row = """
-- TODO: Replace with your actual query to check a specific row or aspect of the CTE result.
    SELECT
        f.title,
        COUNT(r.inventory_id) AS n_rentals
    FROM film AS f
    JOIN film_category AS fc ON f.film_id = fc.film_id
    JOIN category AS c ON fc.category_id = c.category_id
    JOIN inventory AS i ON f.film_id = i.film_id
    JOIN rental AS r ON i.inventory_id = r.inventory_id
    WHERE r.rental_date LIKE '%-05-%'
    GROUP BY f.title
    ORDER BY n_rentals DESC
    LIMIT 10;
"""
run_sql(check_cte_row)

,title,n_rentals
0,IDOLS SNATCHERS,5
1,BUCKET BROTHERHOOD,5
2,ROBBERS JOON,5
3,BOOGIE AMELIE,4
4,TALENTED HOMICIDE,4
5,LOVE SUICIDES,4
6,CLOSER BANG,4
7,ENEMY ODDS,4
8,GRIT CLOCKWORK,4
9,PACIFIC AMISTAD,4


> **_Describe your validation and what you concluded here. Include helper queries in code cells if needed._**
> Checked if number of rentals in May CTE matched with original query. Since it does, the CTE works as intended.

---

## 5. Window Function with `OVER (...)` (15 points)

Next, you will write at least one query that uses a **window function**. Options include:
- `ROW_NUMBER()`, `RANK()` or `DENSE_RANK()` within a category (e.g., top customers per store).
- A percent-of-total calculation using `SUM(...) OVER ()` or `SUM(...) OVER (PARTITION BY ...)`.

Your window query should:
- Use `OVER (...)` in a meaningful way.
- Produce a result that could help your audience understand ranking or relative importance.
- Include at least one validation step (e.g., verify that percentages sum to ~100%).


### 5.1 Window function design

Describe your plan:
- What are you ranking or computing percentages over?
- What does each row represent in the final result?
- How will your audience interpret the window column(s)?


> **_Write your window function plan here._**
> Percentage of rentals per film in May. Each row represents a film and its number of rentals and percentage of total rentals for that film. The audience will see the window column as an indicator of what's popular in May.

### 5.2 Writing the window function query

Now that you have a plan for your window function, write the SQL query that implements it.

In [12]:
# 5.2 Window function query

# Now that you've designed your window function query, write the SQL to
# implement it. Make sure to use clear aliases for tables and columns, to
# structure the query in a way that breaks down the logic into understandable
# steps, and to include the appropriate window function(s) with meaningful
# OVER(...) clauses.

window_query = """
-- TODO: Replace with your window function query.
SELECT DISTINCT
    f.title,
    SUM(1) OVER (PARTITION BY f.film_id) AS n_rentals,
    SUM(1) OVER () AS total_rentals,
    SUM(1) OVER (PARTITION BY f.film_id) / SUM(1) OVER () AS pct_of_total
FROM film AS f
JOIN inventory AS i ON f.film_id = i.film_id
JOIN rental AS r ON r.inventory_id = i.inventory_id
WHERE r.rental_date LIKE '%-05-%';
"""

# Once you've defined your query, you can uncomment the lines below to run it and see the results.
df_window = run_sql(window_query)
display(df_window) or df_window.head()

,title,n_rentals,total_rentals,pct_of_total
0,ACADEMY DINOSAUR,2,1156,0.0017
1,ADAPTATION HOLES,1,1156,0.0009
2,AFFAIR PREJUDICE,2,1156,0.0017
3,AFRICAN EGG,1,1156,0.0009
4,AGENT TRUMAN,2,1156,0.0017
...,...,...,...,...
681,WYOMING STORM,2,1156,0.0017
682,YENTL IDAHO,2,1156,0.0017
683,ZHIVAGO CORE,1,1156,0.0009
684,ZOOLANDER FICTION,1,1156,0.0009


,title,n_rentals,total_rentals,pct_of_total
0,ACADEMY DINOSAUR,2,1156,0.0017
1,ADAPTATION HOLES,1,1156,0.0009
2,AFFAIR PREJUDICE,2,1156,0.0017
3,AFRICAN EGG,1,1156,0.0009
4,AGENT TRUMAN,2,1156,0.0017


### 5.3 Window function validation

Identify at least one validation you can perform, such as:
- Summing `pct_of_total` and confirming it is close to 100 (allowing for rounding).
- Checking that the ranking order matches the sorted totals.
- Comparing one row’s percent-of-total to a hand calculation.

In [13]:
# As appropriate, write helper queries or pandas checks for window function results
# e.g., df_window['pct_of_total'].sum()
df_window['pct_of_total'].sum()

Decimal('1.0070')

> **_Write your validation explanation here and include helper code if needed._**
> Summed the percentages and rounded to confirm the percentages add up to 100 (or very closely to). It adds up to 1.0070, which rounds down to 1, so this window function worked as intended.

---

## 6. Performance Check with `EXPLAIN` and/or Index (5 points)

Pick **one** of your more complex queries from the earlier sections (join, aggregation, or window-based) and perform a brief performance analysis using MySQL's `EXPLAIN` statement.

You should:
- State which query you are examining and what aspect of its execution plan you want to inspect.
- Run `EXPLAIN` on the query and capture the output in a DataFrame.
- Identify at least one aspect of the plan (e.g., table scan vs. index usage, join order, estimated rows) and explain why it seems reasonable or concerning.
- Optionally, propose or implement a simple index and compare the `EXPLAIN` output before vs. after.

You do **not** need to time queries precisely or chase microseconds. The focus is on reading and interpreting the query plan.

**Note**: You might not see any "red flags" in the `EXPLAIN` output, especially on a small dataset like Sakila. That's totally fine! The point is to practice reading the plan and thinking about what it means.

### 6.1 Choose a query and plan your analysis

Before running `EXPLAIN`, take a moment to describe your plan:
1. Which query from an earlier section are you examining (e.g., "Section 3 aggregation by category", "Section 5 window query")?
2. Why did you choose this query (e.g., it joins many tables, it aggregates a large dataset)?
3. What do you expect to see in the `EXPLAIN` output (e.g., index usage on join keys, a full scan on a particular table)?

> **_Write your plan here._**
> Section 3 aggregation by category. I chose this query because it has a couple of joins and it has to go over every film that's associated with a category. I expect to see index usage on join keys, a note about using a filter (where statement), and const/eq_ref for access type for joined tables.

### 6.2 Run EXPLAIN on your chosen query

Now that you have a plan, run `EXPLAIN` on the query you chose. Note that you can prefix any SQL query with `EXPLAIN` to get the execution plan — `run_sql()` works with `EXPLAIN` statements just like any other query.

In [14]:
# 6.2 Run EXPLAIN on your chosen query

# Prefix your query with EXPLAIN to capture the execution plan.
# Note: run_sql() works with EXPLAIN statements just like any other query.

explain_query = f"""
-- TODO: Replace with EXPLAIN for one of your actual queries from earlier sections.
EXPLAIN {agg_query}
"""

# Once you've defined your query, you can uncomment the lines below to run it and see the results.
df_explain = run_sql(explain_query)
# display(df_explain) or df_explain.head()
display(df_explain)

,id,select_type,table,partitions,type,possible_keys,key,key_len,ref,rows,filtered,Extra
0,1,SIMPLE,c,None,ALL,PRIMARY,NaN,NaN,NaN,16,100.0,Using temporary; Using filesort
1,1,SIMPLE,fc,None,ref,"PRIMARY,fk_film_category_category",fk_film_category_category,1,sakila.c.category_id,62,100.0,Using index
2,1,SIMPLE,f,None,eq_ref,PRIMARY,PRIMARY,2,sakila.fc.film_id,1,100.0,NaN


### 6.3 Interpret the EXPLAIN output

Now that you have the `EXPLAIN` output, interpret it in a few sentences. Address at least one of the following:
- Is MySQL using an index or doing a full table scan on any of the tables? Does this seem reasonable given the query structure?
- What is the estimated number of rows being scanned for each table (the `rows` column)? Does this seem appropriate?
- Is the join order what you would expect? Why or why not?
- Based on the plan, would you suggest any changes to improve performance (e.g., adding an index, restructuring the query)?

In [15]:
# Optional: use this cell for any helper queries or pandas checks that
# support your interpretation of the EXPLAIN output.
# For example, you could check the actual row count of a table where EXPLAIN
# shows a large row estimate.
helper_query = """
SELECT COUNT(*) FROM category;
"""
df_helper = run_sql(helper_query)
display(df_helper)

,COUNT(*)
0,16


> **_Write your performance interpretation here (2–5 sentences)._**
> MySQL is using an index for film_category and a primary key lookup for film, but is doing a full table scan on category. It makes sense since the aggregated query is going over all 16 categories (ergo the full table scan), looking for a film that's a member of a particular category (ergo ref; multiple rows result from index lookup), and computes the length of a specific film into the calculation (ergo eq_ref; single row result from primary key lookup).
>
> The tables' number of row scans makes sense since, for category, there are 16 in total, for film_category, films can only have one category (62 < 1000; many rows can get scanned but the number is less than the total number of films), and for film, films are picked individually to add into the calculation.

### 6.4 Optional: Propose or add an index

If you'd like to go further and it seems like an optimization is needed, you can propose or create a simple index and compare the `EXPLAIN` output before vs. after. Document:
- Which column(s) you chose and why (e.g., frequently used in `WHERE` or join conditions).
- Whether the `EXPLAIN` output changed in a way that suggests better performance (e.g., less full-table scanning, lower estimated rows).

In [16]:
# Optional: create an index and re-run EXPLAIN to compare.
# create_index_sql = """
# -- TODO: Replace with your CREATE INDEX statement.
# -- Example: CREATE INDEX idx_rental_date ON rental(rental_date);
# """
# run_sql(create_index_sql)

# explain_after_index = """
# -- TODO: Replace with your EXPLAIN query after adding the index.
# """
# run_sql(explain_after_index)

---

## 7. Manager-Style Report Table (10 points)

Design **one table** that you would feel comfortable showing to your imagined audience or including in a portfolio. This table should:
- Be relatively **wide** (e.g., 4–8 columns) with clear, human-readable column labels.
- Summarize something decision-relevant (e.g., categories by revenue and share, top customers per store).
- Optionally use `CASE` expressions or window functions to create computed or pivot-like columns.

You may reuse or adapt one of your earlier queries if it already reads like a manager report, or you may write a dedicated query here.

### 7.1 Describe your report

Before writing the SQL, describe your report:
1. Who is the audience for this table (e.g., store manager, marketing analyst, operations lead)?
2. What decision or question does this table support?
3. Which columns will appear, and what does each one tell the audience?
4. Will you reuse a query from an earlier section or write a new one?

> **_Write your audience, decision, and column plan here._**
> The audience is the store manager. This table supports the question: "what films make up the majority of rentals in May?" There are 4 columns: title (the name of the film), n_rentals_in_may (the number of rentals of a film in May), total_rentals_in_may (the total number of rentals of all films in May), and pct_of_total_rentals_in_may (the percentage of the total number of rentals that a film consists of).
> I will reuse the Section 5 window query and add a order by statement to make the more rented out films show up on top.

### 7.2 Write the report query

Now write the SQL query that produces your report table. Aim for clear, human-readable column aliases and a layout that could stand on its own in a portfolio or presentation.

In [17]:
# 7.2 Report query SQL

# Now that you've described your report, write the SQL that produces it.
# Aim for clear column aliases and a result that could stand on its own in
# a portfolio or presentation.

report_query = """
-- TODO: Replace with your manager-style report query.
-- Your report should have 4–8 columns with human-readable labels.
-- You may adapt a query from an earlier section or write a new one here.
SELECT DISTINCT
    f.title,
    SUM(1) OVER (PARTITION BY f.film_id) AS n_rentals_in_may,
    SUM(1) OVER () AS total_rentals_in_may,
    SUM(1) OVER (PARTITION BY f.film_id) / SUM(1) OVER () AS pct_of_total_rentals_in_may
FROM film AS f
JOIN inventory AS i ON f.film_id = i.film_id
JOIN rental AS r ON r.inventory_id = i.inventory_id
WHERE r.rental_date LIKE '%-05-%'
ORDER BY pct_of_total_rentals_in_may DESC;
"""

# Once you've defined your query, you can uncomment the lines below to run it and see the results.
df_report = run_sql(report_query)
display(df_report) or df_report.head()

,title,n_rentals_in_may,total_rentals_in_may,pct_of_total_rentals_in_may
0,BUCKET BROTHERHOOD,5,1156,0.0043
1,IDOLS SNATCHERS,5,1156,0.0043
2,ROBBERS JOON,5,1156,0.0043
3,BOOGIE AMELIE,4,1156,0.0035
4,CLOSER BANG,4,1156,0.0035
...,...,...,...,...
681,WONDERLAND CHRISTMAS,1,1156,0.0009
682,WORST BANGER,1,1156,0.0009
683,WRATH MILE,1,1156,0.0009
684,ZHIVAGO CORE,1,1156,0.0009


,title,n_rentals_in_may,total_rentals_in_may,pct_of_total_rentals_in_may
0,BUCKET BROTHERHOOD,5,1156,0.0043
1,IDOLS SNATCHERS,5,1156,0.0043
2,ROBBERS JOON,5,1156,0.0043
3,BOOGIE AMELIE,4,1156,0.0035
4,CLOSER BANG,4,1156,0.0035


### 7.3 Optional: polish in pandas

If you wish, you can use `pandas` to make the report output more presentation-ready:
- Rename columns for readability.
- Format numeric columns (e.g., round percentages, add currency formatting).
- Select or reorder columns for clarity.

This is optional but can help the output feel more portfolio-ready.

In [18]:
# Optional: light pandas polishing for the report table, then display it.
# For example, you could format the revenue column as currency, or add a total row at the bottom.

# Put your Pandas commands to polish the report with pandas here (example: format revenue as currency):
# If I gave myself more time for this assignment, I would change column names from snake case to plain English, 
# and convert pct_of_total_rentals_in_may from string to percentage

# Display the polished report
# display(df_report)

---

## 8. Reflection on SQL Reasoning and Debugging (5 points)

Write **1–2 paragraphs** reflecting on your process. Address at least:
- How you broke down your mini-project into smaller SQL questions, or if your mini-project emerged/evolved as you explored the data and wrote individual queries.
- Any bugs, confusing results, or dead ends you hit and how you debugged them (e.g., row-count checks, `LIMIT`, CTE inspection, switching between Adminer and this notebook).
- Any tradeoffs you noticed between expressing logic with CTEs, `GROUP BY`, and window functions (e.g., readability vs. performance, ease of validation) either for this specific assignment or in general based on your experience.


> **_Write your reflection here._**
> I started off decomposing the mini-project question by coming up with questions about the data that might pertain to the mini-project question. However, I realized that my question was too broad as it was hard to come up with smaller SQL questions that could add up to the main question. I ended up rewriting my question to fit the idea I was starting to brew through my exploration of the data via querying. While querying, I ran into many deadends and confusing results. Most deadends were a result of syntax errors (commas in the select fields and where statements past the group by statement), which I resolved by comparing the current query with queries from assignments in the past. The confusing result I had was with my initial solution for the window query, in which many rows seemed duplicated, and with the explain query. The window query had duplicated records because it doesn't aggregate the data (which I didn't know too well as I didn't have much practice with it and forgot), so I decided to add the distinct option to the select statement to make it more readable. As for the explain query, I had to reference the day 19 ica as I couldn't make heads of the output upon first glance. Tradeoffs I noted between expressing logic with CTEs, GROUP BY, and window functions were that CTEs made the query itself more accessible but tanked in performance whereas group by and window functions required more thought to implement (and in the latter case, understand the output) but performed quickly. It should be noted that the nature of the queries were all relatively different from one another, so it's hard to point out whether the performance was due to how the different logic expressions worked or if the data simply required more or less effort.

---

## 9. AI Audit Log (5 points)

This section is about **transparent, professional use of AI tools** (including systems like ChatGPT, GitHub Copilot, Perplexity, or others). You will not lose points for using AI, but you **must** document how you used it.

Answer the prompts below honestly. If you did **not** use any AI tools, you can say so explicitly.


### 9.1 Tools used

- List any AI tools you used while working on this assignment (e.g., ChatGPT, Copilot, Perplexity, Claude, etc.). If none, write "None".
- For each tool, briefly describe what you asked it to help with (e.g., "helped me remember `CREATE INDEX` syntax", "suggested a pattern for a window function").

> **_Describe your AI usage here._** None

### 9.2 Directly used vs. adapted

- Did you paste any AI-generated SQL or code directly into your notebook with minimal changes? If so, point to where (e.g., "Section 5 window query") and **make sure you add a comment in the code cell acknowledging the AI contribution**.
- For code or queries that you **adapted**, briefly describe what you changed and why (e.g., "adjusted table names to match Sakila, simplified to match our validation habits").

> **_Describe AI "copy-paste" usage here._** x

### 9.3 Validation and trust

- How did you validate any AI-suggested queries or patterns before trusting them? (e.g., row-count checks, comparing against a simpler query, sanity checking outputs.)
- Are there any parts of your notebook where you are **not fully confident** in the result? If so, describe them, why you're lacking confidence, and what additional checks you would run if you had more time (e.g., "I wasn't sure if the window function was partitioned correctly, I would want to double-check the logic and maybe test it on a smaller dataset if I had more time").

> **_Describe your trust in AI outputs here._** x

- - -

## 10. Creating your GitHub repo and pushing your code (10 points)

Now that you've finished your notebook, it's time to create a GitHub repository and push your code. 

**The goal for this section is to build a repository that someone could clone, spin up the Docker container to access the SQL database, and execute your notebook from top to bottom.**

To achieve this, follow these steps:

1. Create a new private repository on GitHub named `cmse492-hw02-yourMSUNetID` (replace `yourMSUNetID` with your actual MSU NetID).
2. Initialize the repository with a README file.
3. Clone the repository to your local machine.
4. Copy your completed notebook into the cloned repository folder. **Commit the fully-executed version so that someone could see the output _without_ having to run it, but using the Docker set up they should be able to execute it, if needed.**
5. Add the necessary Docker compose file needed to run the Sakila database (you can reuse the one provided in the course materials).
6. Update the README so that it provides an overview of the project, what you accomplished and the skills you're showcasing, and instructions on how to run the notebook and connect to the database (make sure you include information for starting up the Docker containers!).

    - You should be able to share this repository with a friend, co-worker, or future employer and they should be able to understand what you did, why you did it, and how to run your notebook and see your analysis in action.
    
7. Use Git commands to add, commit, and push your changes to GitHub. 

- - -
After you're made sure the GitHub repo is set up correctly, **Submit this notebook to D2L** and **Fill out the MS Form below** to finalize your submission.  
(If the form won't render in your notebook for some reason, you can also access it directly here: https://forms.office.com/r/Ppk0tnVkc8)

In [20]:
from IPython.display import HTML
HTML(
"""
<iframe 
	src="https://forms.office.com/r/Ppk0tnVkc8" 
	width="800" 
	height="800px" 
	frameborder="0" 
	marginheight="0" 
	marginwidth="0">
	Loading...
</iframe>
"""
)

- - -
### Congratulations, you're done!

Submit this assignment by uploading it to the course Desire2Learn web page.  Go to the "Homework" section, find the appropriate submission link, and upload it there. **Make sure your instructors have access to your private GitHub repo by adding them as collaborators!**

See you in class!

---
&#169; Copyright 2026, The Department of Computational Mathematics, Science and Engineering at Michigan State University.